<a href="https://colab.research.google.com/github/Janai10/FirstWebsit/blob/main/Task_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, shutil

# Detects Colab vs local Jupyter and picks the right folder. No editing needed.
IN_COLAB = os.path.isdir("/content")
base     = "/content" if IN_COLAB else os.path.expanduser("~")
project  = os.path.join(base, "mapreduce_lab")

if os.path.exists(project):
    shutil.rmtree(project)
os.makedirs(os.path.join(project, "input"))
os.chdir(project)

print("environment :", "Google Colab" if IN_COLAB else "local Jupyter")


environment : Google Colab


In [ ]:
%pip install -q datasets pyspark

import json
from datasets import load_dataset
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, trim

print('='*100)
print('ANSWERSHEET SECTION 1.2 & EP1: BIG DATA JUSTIFICATION, 5Vs, SCHEMA & LICENSE')
print('='*100)
print('Target Report Section   : AnswerSheet Section 1.2 (Big Data Justification) & Section EP1 (Evidence Pack)')
print('Target Repository       : ikimij2_16888207')
print('='*100)

# 1. Initialize PySpark Session (Fixed line indentation using parentheses)
spark = (
    SparkSession.builder
    .appName('Day01_1_GQA_Question_Type_Classification')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '12')
    .getOrCreate()
)

print('\n[STATUS] SparkSession Initialized Successfully!')
print(f'[INFO] Spark Version       : {spark.version}')
print(f'[INFO] Master Node         : {spark.sparkContext.master}')


# 2. Stream the FULL GQA training split in batches
print('\n' + '=' * 100)
print('FULL DATASET ACCESS & VERIFICATION (EP1)')
print('=' * 100)

gqa_stream = load_dataset(
    'lmms-lab-encoder/GQA',
    'train_all_instructions',
    split='train',
    streaming=True
)

# Read the first record to verify access
first_record = next(iter(gqa_stream))

print("[STATUS] HUGGING FACE ACCESS VERIFIED")
print("Dataset: lmms-lab-encoder/GQA")
print("Configuration: train_all_instructions")
print("Split: train")
print("Streaming access: successful")
print("\nFirst record fields:")
print(list(first_record.keys()))

print("\nFirst record preview:")
print(json.dumps(first_record, indent=2, default=str)[:2000])

# Read the complete extracted dataset into Spark.
df = spark.read.parquet(output_path)

print('[INFO] Dataset Source       : Hugging Face')
print('[INFO] Dataset              : GQA')
print('[INFO] Dataset Subset       : train_all_instructions')
print('[INFO] Split                 : train')
print('[INFO] Access Method         : Streaming in batches')
print('[INFO] Full Dataset Download : No full in-memory download')
print(f'[INFO] Records Scanned       : {total_scanned:,}')
print(f'[INFO] Records Written       : {total_written:,}')
print(f'[INFO] Parquet Output        : {output_path}')

print('[STATUS] Full streamed dataset saved and loaded into Spark.')
print('=' * 100)

# 3. The 5 V\'s of Big Data Empirical Verification Table
print('\n' + '=' * 100)
print('EMPIRICAL VERIFICATION: THE 5 Vs OF BIG DATA (GQA)')
print('=' * 100)
print(f'[1] VOLUME   : {df.count():,} records loaded into Spark.')
print('             These records are from the selected training split.')
print('[2] VELOCITY : Data is accessed through streaming from Hugging Face.')
print('             This does not establish real-time data generation speed.')
print('[3] VARIETY  : Inspect question, answer, image-related and other fields')
print('             using the schema below. Some nested fields are stored as JSON.')
print('[4] VERACITY : Missing values and duplicate IDs are checked below.')
print('             These checks apply only to the loaded training split.')
print('[5] VALUE    : Question type classification may help categorise questions.')
print('             Target labels must be verified before model training.')
print('=' * 100)

# 4. License and Ethical Compliance Verification
print('\n' + '=' * 100)
print('DATASET LICENSE & ETHICAL COMPLIANCE VERIFICATION')
print('=' * 100)
print('[LICENSE] Dataset Source : Hugging Face - lmms-lab-encoder/GQA')
print('[LICENSE] License Status : Free - no registration.')
print('[ETHICS] Review answer and question data usage conditions.')
print('[ETHICS] Check for class imbalance, data bias and classification errors.')
print('[ETHICS] Follow applicable attribution and reuse requirements.')
print('='*100)

# 5. Load DataFrame and Print Schema
print('\n' + '=' * 100)
print('DATASET STRUCTTYPE SCHEMA PRINT (EP1 EVIDENCE PACK)')
print('=' * 100)

df.printSchema()

print('\n[DATA PREVIEW] First five records:')
df.show(5, truncate=False)

print(f'\n[SUMMARY] Records Loaded : {df.count():,}')
print(f'[SUMMARY] Total Columns         : {len(df.columns)}')
print(f'[SUMMARY] RDD Partitions        : {df.rdd.getNumPartitions()}')

# 6. Check missing or blank values
print('\n' + '=' * 100)
print('MISSING VALUE VERIFICATION (FULL TRAINING SPLIT)')
print('=' * 100)

missing_values = df.select([
    sum(
        when(
            col(c).isNull() |
            (trim(col(c).cast('string')) == ''),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)

# 7. Check duplicate IDs, if the ID column exists
print('\n' + '=' * 100)
print('DUPLICATE ID VERIFICATION (FULL TRAINING SPLIT)')
print('=' * 100)

if 'id' in df.columns:
    duplicate_ids = (
        df.groupBy('id')
        .count()
        .filter(col('count') > 1)
    )

    duplicate_ids.show(truncate=False)
    print(f'[SUMMARY] Duplicate ID Groups: {duplicate_ids.count():,}')
else:
    print('[INFO] No id column found; duplicate ID check skipped.')

# 8. Inspect available question type information
print('\n' + '=' * 100)
print('QUESTION TYPE CLASSIFICATION: LABEL INSPECTION')
print('=' * 100)

if 'question' in df.columns:
    df.select('question').show(10, truncate=False)
else:
    print('[INFO] No question column found. Inspect the schema above.')

if 'types' in df.columns:
    df.select('types').show(10, truncate=False)
else:
    print('[INFO] No types column found. Check available schema fields.')

print('=' * 100)
print('[SUB-TASK 1.2] FULL TRAINING SPLIT DATASET VERIFICATION COMPLETE')
print('[SUB-TASK EP1] FULL TRAINING SPLIT SCHEMA AND QUALITY CHECKS COMPLETE')
print('[NOTE] Results describe the records successfully streamed and written.')
print('[NOTE] Confirm the stream completed without errors before reporting full-split results.')
print('=' * 100)

ANSWERSHEET SECTION 1.2 & EP1: BIG DATA JUSTIFICATION, 5Vs, SCHEMA & LICENSE
Target Report Section   : AnswerSheet Section 1.2 (Big Data Justification) & Section EP1 (Evidence Pack)
Target Repository       : ikimij2_16888207

[STATUS] SparkSession Initialized Successfully!
[INFO] Spark Version       : 4.0.4
[INFO] Master Node         : local[*]

FULL DATASET ACCESS & VERIFICATION (EP1)
[PROGRESS] Scanned: 5,000 | Written: 5,000
[PROGRESS] Scanned: 10,000 | Written: 10,000
[PROGRESS] Scanned: 15,000 | Written: 15,000
[PROGRESS] Scanned: 20,000 | Written: 20,000
[PROGRESS] Scanned: 25,000 | Written: 25,000
[PROGRESS] Scanned: 30,000 | Written: 30,000
[PROGRESS] Scanned: 35,000 | Written: 35,000
[PROGRESS] Scanned: 40,000 | Written: 40,000
[PROGRESS] Scanned: 45,000 | Written: 45,000
[PROGRESS] Scanned: 50,000 | Written: 50,000
[PROGRESS] Scanned: 55,000 | Written: 55,000
[PROGRESS] Scanned: 60,000 | Written: 60,000
[PROGRESS] Scanned: 65,000 | Written: 65,000
[PROGRESS] Scanned: 70,000 |